# 01. Exploratory Data Analysis (EDA) & Image Preprocessing for Vision Track

**Project:** Durian Ripeness Classification via Cross-Modal Knowledge Distillation (CMKD)  
**Responsible:** Member 1 (Vision Lead)  
**Reference Documents:** `README.md` (Dataset specification) and `specification/Đặc tả đề tài.pdf` (Section 1.3.1 - Vision Track)

---

### Notebook Objectives

This notebook serves as the testbed to inspect vision image data and **experiment with, validate, and benchmark all preprocessing routines**. Once all pipeline steps are proven robust here, the `VisionPreprocessor` class is replicated into [preprocess_vision.py](preprocess_vision.py) for downstream training notebooks to import and reuse.

### Notebook Structure

| Section | Content |
| :--- | :--- |
| 1 | Library installation, imports, and global configuration |
| 2 | Google Drive connection, path declaration, and data integrity checks |
| 3 | Ripeness label distribution, train/val/test splits, and correlation with physicochemical metrics |
| 4 | 6-viewpoint camera angle analysis and per-fruit image completeness |
| 5 | Image quality audit: resolution, brightness, contrast, and sharpness |
| 6 | Preprocessing and data augmentation: CLAHE, policy-based augmentation, Mixup/CutMix |
| 7 | PyTorch Dataset and DataLoader encapsulation |
| 8 | Conclusions and deliverables |

> ⚠️ **Data Safety Principle:** This notebook strictly reads data from `data/raw/` and `data/split_data/`. All processing operations occur in memory; raw files are never overwritten or modified.

## 1. Library Installation and General Configuration

In [ ]:
# Install missing libraries (primarily needed when running on Colab)
import importlib.util
import subprocess
import sys

REQUIRED_PACKAGES = {  # pip package name -> import module name
    "opencv-python": "cv2",
    "pillow": "PIL",
    "torch": "torch",
    "torchvision": "torchvision",
    "pandas": "pandas",
    "numpy": "numpy",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "tqdm": "tqdm",
}

missing_packages = [pkg for pkg, module in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing_packages:
    print(f"[INFO] Installing missing packages: {', '.join(missing_packages)}")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing_packages])

print("[SUCCESS] All required libraries are ready.")

In [ ]:
# Import libraries, set random seed, and define global constants
import glob
import os
import random
import re
import warnings
from typing import Dict, List, Optional, Sequence, Tuple, Union

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torchvision.transforms as transforms
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm

try:
    from IPython.display import display
except ImportError:
    display = print

warnings.filterwarnings("ignore")

# Fix random seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Plot configuration
plt.style.use("seaborn-v0_8-whitegrid" if "seaborn-v0_8-whitegrid" in plt.style.available else "default")
plt.rcParams["font.sans-serif"] = ["DejaVu Sans", "Arial", "sans-serif"]
plt.rcParams["axes.unicode_minus"] = False

# Global constants for notebook
RIPENESS_ORDER = ["Unripe", "Ripe", "Overripe"]
RIPENESS_PALETTE = {"Unripe": "#2ecc71", "Ripe": "#f39c12", "Overripe": "#e74c3c", "Unknown": "gray"}
LABEL_TO_IDX = {name: idx for idx, name in enumerate(RIPENESS_ORDER)}
IDX_TO_LABEL = {idx: name for name, idx in LABEL_TO_IDX.items()}
ORIENTATIONS = ["TOP", "BOT", "SA", "SB", "SC", "SD"]
IMAGE_EXTENSIONS = (".jpg", ".jpeg", ".png")


def list_images(folder: str) -> List[str]:
    """List images in a directory, case-insensitive and without duplicates on Windows."""
    if not os.path.isdir(folder):
        return []
    return sorted(
        os.path.join(folder, name)
        for name in os.listdir(folder)
        if name.lower().endswith(IMAGE_EXTENSIONS)
    )


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Compute device: {DEVICE}")
if DEVICE.type == "cuda":
    print(f"[INFO] GPU: {torch.cuda.get_device_name(0)}")
else:
    print("[WARNING] Running on CPU. GPU is recommended on Colab for processing large image sets.")

## 2. Google Drive Connection and Data Verification

Data is hosted on Google Drive as specified in `README.md`:
- `data/raw/` directory: [Drive link](https://drive.google.com/drive/folders/1gIhGj-txv1ZF_RfslbAh7bagNZdnIRhL?usp=drive_link)
- `data/split_data/` directory: [Drive link](https://drive.google.com/drive/folders/1ro-ETqoDNCuDwbYdZkvDuztLAF5B0XTC?usp=drive_link)

The cell below performs:
1. Automatically mounts Google Drive when running in Colab.
2. Defines `DATA_RAW_DIR` and `DATA_SPLIT_DIR` paths (modify these if data is located elsewhere).
3. Verifies mandatory files and directories; halts and lists any missing items if absent.
4. If image directory is empty (e.g. initial local clone), creates a small mock image set in temporary cache so subsequent cells can execute.

In [ ]:
# 2.1. Mount Google Drive when running on Colab
IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import drive

    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
    print("[SUCCESS] Google Drive connected at /content/drive")
else:
    print("[INFO] Running on local environment.")

# 2.2. Declare data paths (modify here if data is located elsewhere)
if IN_COLAB:
    DATA_RAW_DIR = "/content/drive/MyDrive/durian-ripeness-cmkd/data/raw"
    DATA_SPLIT_DIR = "/content/drive/MyDrive/durian-ripeness-cmkd/data/split_data"
    DEMO_CACHE_DIR = "/content/colab_temp_cache/demo_vision_samples"
else:
    DATA_RAW_DIR = os.path.abspath("../../../data/raw")
    DATA_SPLIT_DIR = os.path.abspath("../../../data/split_data")
    DEMO_CACHE_DIR = os.path.abspath("demo_vision_samples")

CSV_RAW_PATH = os.path.join(DATA_RAW_DIR, "durian_characteristics_cleaned.csv")
CSV_SPLIT_MAPPING = os.path.join(DATA_SPLIT_DIR, "sample_split_mapping.csv")
RGB_RAW_DIR = os.path.join(DATA_RAW_DIR, "dataset_clean_rgb")

# 2.3. Check mandatory files and directories
required_items = [
    ("Raw data directory", DATA_RAW_DIR, os.path.isdir),
    ("Split data directory", DATA_SPLIT_DIR, os.path.isdir),
    ("Characteristics CSV", CSV_RAW_PATH, os.path.isfile),
    ("Split mapping CSV", CSV_SPLIT_MAPPING, os.path.isfile),
    ("RGB image directory", RGB_RAW_DIR, os.path.isdir),
]

print("\n[INFO] Checking data paths:")
missing_items = []
for name, path, check in required_items:
    ok = check(path)
    print(f"  {'[OK]' if ok else '[MISSING]'} {name:<22}: {path}")
    if not ok:
        missing_items.append(name)

if missing_items:
    raise FileNotFoundError(
        f"Missing {len(missing_items)} required items: {', '.join(missing_items)}. "
        "Please check if Google Drive is mounted and update DATA_RAW_DIR / DATA_SPLIT_DIR."
    )


# 2.4. Scan real images; if absent, generate synthetic demo samples in temp cache
def create_demo_images(folder: str) -> None:
    """Generate synthetic durian images (elliptical shape with spines) adhering to naming convention."""
    os.makedirs(folder, exist_ok=True)
    demo_fruits = [  # (sample_id, maturity, class, ripeness, RGB rind color)
        (1, "Immature", "A", "Unripe", (70, 140, 50)),
        (8, "Immature", "A", "Ripe", (130, 160, 45)),
        (15, "Immature", "A", "Overripe", (165, 145, 55)),
        (45, "Mature", "B", "Ripe", (140, 170, 50)),
    ]
    height, width = 400, 600  # Maintain 3:2 aspect ratio matching original camera
    center = (width // 2, height // 2)

    for sample_id, maturity, cls, ripeness, color in demo_fruits:
        for orientation in ORIENTATIONS:
            path = os.path.join(folder, f"durian_{sample_id:03d}_{maturity}_{cls}_{ripeness}_{orientation}.jpg")
            if os.path.exists(path):
                continue
            canvas = np.full((height, width, 3), (235, 235, 230), dtype=np.uint8)
            axes = (150, 150) if orientation in ("TOP", "BOT") else (140, 170)
            cv2.ellipse(canvas, center, axes, 0, 0, 360, color, -1)

            rng = np.random.RandomState(sample_id * 100 + ord(orientation[0]))
            for _ in range(300):  # Draw spines
                pt = (rng.randint(center[0] - 130, center[0] + 130), rng.randint(center[1] - 160, center[1] + 160))
                if ((pt[0] - center[0]) / axes[0]) ** 2 + ((pt[1] - center[1]) / axes[1]) ** 2 < 0.85:
                    spike = tuple(int(np.clip(c + rng.randint(-30, 30), 0, 255)) for c in color)
                    cv2.circle(canvas, pt, rng.randint(3, 8), spike, -1)
            if orientation == "TOP":  # Draw pedicel
                cv2.circle(canvas, center, 30, (70, 50, 30), -1)
            cv2.imwrite(path, cv2.cvtColor(canvas, cv2.COLOR_RGB2BGR))


real_images = list_images(RGB_RAW_DIR)
if real_images:
    ACTIVE_IMAGE_DIR = RGB_RAW_DIR
    print(f"\n[SUCCESS] Utilizing {len(real_images)} real images from: {ACTIVE_IMAGE_DIR}")
else:
    create_demo_images(DEMO_CACHE_DIR)
    ACTIVE_IMAGE_DIR = DEMO_CACHE_DIR
    print(f"\n[WARNING] Real image directory is empty; falling back to synthetic demo images at: {ACTIVE_IMAGE_DIR}")
    print("   Image statistics below are illustrative only.")

## 3. Ripeness Label Distribution and Data Splits

The target task is 3-class ripeness classification (`Ripeness`):
- `Unripe`: Immature, hard pulp, lacking characteristic aroma.
- `Ripe`: Optimum commercial maturity, sweet and soft pulp, expanded carpel grooves.
- `Overripe`: Senescent stage, softened/mushy pulp, split rind, fermenting aroma.

The dataset contains 189 fruits partitioned at the **fruit sample level**: all images of a given fruit belong strictly to a single partition (`train`, `val`, or `test`), preventing data leakage across splits.

In [ ]:
# 3.1. Read characteristics table, merge split mapping, and summarize label distribution
df_char = pd.read_csv(CSV_RAW_PATH)
df_split = pd.read_csv(CSV_SPLIT_MAPPING)
print(f"[INFO] Characteristics table: {len(df_char)} fruits | Split mapping: {len(df_split)} fruits")

if "sample_id" not in df_char.columns:
    df_char["sample_id"] = df_char["Replicate"].astype(str).str.extract(r"([0-9]+)")[0].astype(int)

df_merged = df_char.merge(df_split[["sample_id", "Split"]], on="sample_id", how="left")

ripeness_counts = df_merged["Ripeness"].value_counts().reindex(RIPENESS_ORDER)
print("\n[INFO] Ripeness Class Distribution:")
for name, count in ripeness_counts.items():
    print(f"  {name:<9}: {count:>3} fruits ({count / len(df_merged):.1%})")

print("\n[INFO] Label Distribution across Splits:")
display(pd.crosstab(df_merged["Split"], df_merged["Ripeness"], margins=True)[RIPENESS_ORDER + ["All"]])

colors = [RIPENESS_PALETTE[name] for name in RIPENESS_ORDER]
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].pie(
    ripeness_counts, labels=ripeness_counts.index, autopct="%1.1f%%", startangle=140,
    colors=colors, wedgeprops={"edgecolor": "white", "linewidth": 2},
)
axes[0].set_title(f"Ripeness Class Proportions ({len(df_merged)} fruits)", fontweight="bold")

pd.crosstab(df_merged["Split"], df_merged["Ripeness"])[RIPENESS_ORDER].plot(
    kind="bar", ax=axes[1], color=colors, edgecolor="black", rot=0,
)
axes[1].set_title("Label Distribution across Train, Val, Test Splits", fontweight="bold")
axes[1].set_xlabel("Split")
axes[1].set_ylabel("Number of Fruits")
axes[1].legend(title="Ripeness")

plt.tight_layout()
plt.show()

In [ ]:
# 3.2. Correlation between Ripeness Labels and Physicochemical Measurements
physico_cols = {
    "Firmness_AVE": "Pulp Firmness (kgf/cm²)",
    "TSS_AVE": "Total Soluble Solids TSS (°Brix)",
    "Dry_Matter_AVE": "Dry Matter Ratio (%)",
}

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (col, label) in zip(axes, physico_cols.items()):
    sns.boxplot(data=df_merged, x="Ripeness", y=col, order=RIPENESS_ORDER, palette=RIPENESS_PALETTE, ax=ax)
    ax.set_title(label, fontweight="bold")
    ax.set_xlabel("")
    ax.set_ylabel(label)
plt.tight_layout()
plt.show()

# Medians computed directly from data
medians = df_merged.groupby("Ripeness")[list(physico_cols)].median().reindex(RIPENESS_ORDER)
print("[INFO] Feature Medians by Ripeness Class:")
display(medians.round(2))

firm_unripe, firm_ripe = medians.loc["Unripe", "Firmness_AVE"], medians.loc["Ripe", "Firmness_AVE"]
tss_unripe, tss_ripe = medians.loc["Unripe", "TSS_AVE"], medians.loc["Ripe", "TSS_AVE"]
print(f"[FINDING] From Unripe to Ripe: Firmness decreases from {firm_unripe:.2f} to {firm_ripe:.2f} kgf/cm² "
      f"(~{firm_unripe / max(firm_ripe, 1e-6):.1f}x reduction), while TSS increases from {tss_unripe:.2f} to {tss_ripe:.2f} °Brix.")

## 4. 6-Viewpoint Camera Angle Analysis

Each fruit is captured from 6 standardized camera orientations:
- `TOP`: Top view, clearly showing pedicel and abscission zone.
- `BOT`: Bottom view, showing the apex and carpel suture junctions.
- `SA`, `SB`, `SC`, `SD`: 4 lateral views at 90° intervals, capturing spine distribution and carpel bulging.

Fruit ID, camera angle, and ripeness labels are extracted from the filename. If labels are absent in the filename, they are retrieved from the characteristics table via `sample_id`.

In [ ]:
# 4.1. Extract metadata from filenames and visualize all 6 camera angles of a sample fruit
ORIENTATION_PATTERN = re.compile(r"(TOP|BOT|SA|SB|SC|SD)", re.IGNORECASE)
ID_PATTERNS = [
    re.compile(r"durian_0*([0-9]+)", re.IGNORECASE),
    re.compile(r"(?:^|[_\-\s\.])0*([0-9]+)(?:[_\-\s\.]|$)"),
]
RIPENESS_PATTERN = re.compile(r"(Unripe|Overripe|Ripe)", re.IGNORECASE)
id_to_ripeness = dict(zip(df_merged["sample_id"], df_merged["Ripeness"]))


def parse_image_filename(path: str) -> Optional[Dict]:
    """Extract sample_id, orientation angle, and ripeness label from filename; returns None if unresolved."""
    fname = os.path.basename(path)
    m_ori = ORIENTATION_PATTERN.search(fname)
    m_id = next((m for m in (p.search(fname) for p in ID_PATTERNS) if m), None)
    if not (m_ori and m_id):
        return None

    sample_id = int(m_id.group(1))
    m_rip = RIPENESS_PATTERN.search(fname)
    ripeness = m_rip.group(1).capitalize() if m_rip else id_to_ripeness.get(sample_id, "Unknown")
    return {
        "filepath": path,
        "filename": fname,
        "sample_id": sample_id,
        "orientation": m_ori.group(1).upper(),
        "ripeness": ripeness,
    }


available_images = list_images(ACTIVE_IMAGE_DIR)
parsed = [parse_image_filename(p) for p in available_images]
unmatched = [os.path.basename(p) for p, meta in zip(available_images, parsed) if meta is None]

df_img = pd.DataFrame([meta for meta in parsed if meta is not None])
df_img = df_img.merge(df_split[["sample_id", "Split"]], on="sample_id", how="left")

print(f"[INFO] Successfully recognized {len(df_img)}/{len(available_images)} images, across {df_img['sample_id'].nunique()} fruits.")
if unmatched:
    print(f"[WARNING] {len(unmatched)} images with unrecognized angle or fruit ID, e.g.:")
    for name in unmatched[:10]:
        print(f"   - {name}")

print("\n[INFO] Image count by camera orientation:")
display(df_img["orientation"].value_counts().reindex(ORIENTATIONS))

# Select fruit with most complete angles for visualization
best_id = df_img.groupby("sample_id")["orientation"].nunique().idxmax()
fruit_imgs = df_img[df_img["sample_id"] == best_id]
fruit_label = fruit_imgs["ripeness"].iloc[0]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for ax, orientation in zip(axes.flatten(), ORIENTATIONS):
    match = fruit_imgs[fruit_imgs["orientation"] == orientation]
    if len(match):
        with Image.open(match.iloc[0]["filepath"]) as img:
            img.draft("RGB", (800, 800))  # Decode at lower resolution for rapid rendering
            ax.imshow(img.convert("RGB"))
        ax.set_title(f"Viewpoint: {orientation}", fontweight="bold")
    else:
        ax.text(0.5, 0.5, f"Missing angle {orientation}", ha="center", va="center", color="gray")
    ax.axis("off")
plt.suptitle(f"6 Camera Viewpoints of Fruit #{best_id} ({fruit_label})", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 4.2. Camera Angle Completeness and Image Count per Fruit

Certain fruits were recorded across multiple imaging sessions. This section verifies whether each fruit has all 6 orientations available and examines image count distributions across fruits, guiding sampling strategies during training (such as balancing image counts per fruit).

In [ ]:
# 4.2. Camera angle count and total image distribution per fruit
fruit_stats = df_img.groupby("sample_id").agg(
    total_images=("orientation", "count"),
    unique_orientations=("orientation", "nunique"),
    ripeness=("ripeness", "first"),
).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

sns.countplot(data=fruit_stats, x="unique_orientations", palette="viridis", ax=axes[0])
axes[0].set_title("Distinct Camera Angles per Fruit", fontweight="bold")
axes[0].set_xlabel("Number of Angles")
axes[0].set_ylabel("Number of Fruits")
for bar in axes[0].patches:
    axes[0].annotate(f"{int(bar.get_height())}", (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                     ha="center", va="bottom", xytext=(0, 3), textcoords="offset points")

sns.boxplot(data=fruit_stats, x="ripeness", y="total_images", order=RIPENESS_ORDER, palette=RIPENESS_PALETTE, ax=axes[1])
axes[1].set_title("Images per Fruit by Ripeness Stage", fontweight="bold")
axes[1].set_xlabel("Ripeness")
axes[1].set_ylabel("Images / Fruit")

plt.tight_layout()
plt.show()

num_full = (fruit_stats["unique_orientations"] == len(ORIENTATIONS)).sum()
print(f"[INFO] {num_full}/{len(fruit_stats)} fruits ({num_full / len(fruit_stats):.1%}) have all 6 camera viewpoints.")
print(f"[INFO] Average {fruit_stats['total_images'].mean():.1f} images/fruit "
      f"(range: {fruit_stats['total_images'].min()} to {fruit_stats['total_images'].max()} images).")

## 5. Image Quality Audit

### 5.1. Resolution, Integrity, Brightness, and Contrast

Evaluated across a random sample of up to 100 images:
1. **File Integrity:** Detect corrupted files that could crash training loops.
2. **Dimensions and Aspect Ratio:** Determine appropriate aspect-preserving resize strategies.
3. **Brightness and Contrast:** Assess illumination variance to calibrate `ColorJitter` ranges.

> Raw images are 24MP ($6000 \times 4000$); reading with PIL `Image.draft` allows fast header/thumbnail inspection in seconds rather than tens of minutes over Drive FUSE.

In [ ]:
# 5.1. Audit image quality across random sample
SAMPLE_EDA_SIZE = min(100, len(available_images))
sample_paths = random.Random(SEED).sample(available_images, SAMPLE_EDA_SIZE)

quality_records, corrupted_files = [], []
for path in tqdm(sample_paths, desc="Auditing image quality"):
    try:
        with Image.open(path) as img:
            width, height = img.size  # Read header directly without full decoding
            img.draft("RGB", (512, 512))
            gray = cv2.cvtColor(np.asarray(img.convert("RGB")), cv2.COLOR_RGB2GRAY)
        quality_records.append({
            "filename": os.path.basename(path),
            "width": width,
            "height": height,
            "aspect_ratio": round(width / height, 3),
            "file_size_mb": round(os.path.getsize(path) / 1024 ** 2, 2),
            "mean_brightness": round(float(gray.mean()), 2),
            "contrast": round(float(gray.std()), 2),
        })
    except Exception as exc:
        corrupted_files.append((os.path.basename(path), str(exc)))

df_quality = pd.DataFrame(quality_records)
print(f"[INFO] Valid images: {len(df_quality)} | Corrupted files: {len(corrupted_files)}")
for name, err in corrupted_files:
    print(f"   [ERROR] {name}: {err}")

display(df_quality.describe().round(2))

brightness = df_quality["mean_brightness"]
print(f"[INFO] Mean brightness ranges from {brightness.min():.0f} to {brightness.max():.0f} "
      f"(coefficient of variation: {brightness.std() / brightness.mean():.1%}).")

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_quality["mean_brightness"], kde=True, color="teal", ax=axes[0])
axes[0].set_title("Mean Brightness Distribution", fontweight="bold")
axes[0].set_xlabel("Brightness (0 - 255)")
sns.histplot(df_quality["contrast"], kde=True, color="darkorange", ax=axes[1])
axes[1].set_title("Contrast Distribution (Grayscale Standard Deviation)", fontweight="bold")
axes[1].set_xlabel("Contrast")
plt.tight_layout()
plt.show()

### 5.2. Rind Color Space and Normalization Statistics

1. **Color Shift during Ripening:** The peel transitions from deep green (chlorophyll-rich in `Unripe`) to yellowish-green or golden-brown (carotenoid accumulation in `Ripe` and `Overripe`).
2. **Hue Channel in HSV:** Decouples chromaticity from illumination (OpenCV scale 0–180: green roughly 35–85, yellowish-orange roughly 10–30). Background is filtered by saturation thresholding.
3. **Green/Red Ratio:** A straightforward metric tracking peel yellowing.
4. **Dataset Channel Mean/Std:** Compared with standard ImageNet statistics (`mean=[0.485, 0.456, 0.406]`, `std=[0.229, 0.224, 0.225]`) to guide normalization configuration.

In [ ]:
# 5.2. Analyze Hue, Green/Red ratio, and compute dataset channel mean/std
filename_to_ripeness = dict(zip(df_img["filename"], df_img["ripeness"]))
color_sample_paths = sample_paths[:60]

color_records, channel_means, channel_stds = [], [], []
for path in color_sample_paths:
    with Image.open(path) as img:
        img.draft("RGB", (448, 448))
        img = img.convert("RGB")
        img.thumbnail((224, 224))
        rgb = np.asarray(img)

    rgb_float = rgb / 255.0
    channel_means.append(rgb_float.mean(axis=(0, 1)))
    channel_stds.append(rgb_float.std(axis=(0, 1)))

    hsv = cv2.cvtColor(rgb, cv2.COLOR_RGB2HSV)
    fruit_mask = hsv[:, :, 1] > 25  # Filter low-saturation white/gray background
    if fruit_mask.sum() > 50:
        color_records.append({
            "ripeness": filename_to_ripeness.get(os.path.basename(path), "Unknown"),
            "mean_hue": float(hsv[:, :, 0][fruit_mask].mean()),
            "green_red_ratio": float(rgb_float[:, :, 1][fruit_mask].mean() / (rgb_float[:, :, 0][fruit_mask].mean() + 1e-6)),
        })

df_colors = pd.DataFrame(color_records)
if df_colors["ripeness"].nunique() > 1:
    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    sns.boxplot(data=df_colors, x="ripeness", y="mean_hue", order=RIPENESS_ORDER, palette=RIPENESS_PALETTE, ax=axes[0])
    axes[0].set_title("Rind Hue by Ripeness Class", fontweight="bold")
    axes[0].set_ylabel("Hue (0 - 180, OpenCV)")
    axes[0].set_xlabel("Ripeness")
    sns.boxplot(data=df_colors, x="ripeness", y="green_red_ratio", order=RIPENESS_ORDER, palette=RIPENESS_PALETTE, ax=axes[1])
    axes[1].set_title("Green/Red Ratio by Ripeness Class", fontweight="bold")
    axes[1].set_ylabel("Green / Red")
    axes[1].set_xlabel("Ripeness")
    plt.tight_layout()
    plt.show()
    display(df_colors.groupby("ripeness")[["mean_hue", "green_red_ratio"]].median().reindex(RIPENESS_ORDER).round(3))

# Compare normalization statistics against ImageNet
DATASET_MEAN = np.mean(channel_means, axis=0)
DATASET_STD = np.mean(channel_stds, axis=0)
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD = np.array([0.229, 0.224, 0.225])

df_norm = pd.DataFrame(
    {"Dataset mean": DATASET_MEAN, "ImageNet mean": IMAGENET_MEAN, "Dataset std": DATASET_STD, "ImageNet std": IMAGENET_STD},
    index=["R", "G", "B"],
).round(4)
display(df_norm)

max_mean_gap = float(np.abs(DATASET_MEAN - IMAGENET_MEAN).max())
print(f"[INFO] DATASET_MEAN = {DATASET_MEAN.round(4).tolist()}")
print(f"[INFO] DATASET_STD  = {DATASET_STD.round(4).tolist()}")
if max_mean_gap < 0.05:
    print(f"[NOTE] Dataset mean is close to ImageNet (max gap {max_mean_gap:.3f}); using ImageNet normalization is suitable.")
else:
    print(f"[NOTE] Dataset mean deviates from ImageNet (max gap {max_mean_gap:.3f}). "
          "When fine-tuning pretrained backbones, retain ImageNet normalization to align with learned feature weights; "
          "pass DATASET_MEAN / DATASET_STD to VisionPreprocessor if training from scratch.")

### 5.3. Sharpness and Thorn Edge Density

Carpel suture grooves and spine microstructures provide critical visual cues for ripeness; hence blur verification is essential:
1. **Laplacian Variance** $\mathrm{Var}(\nabla^2 I)$: Sharp images with dense spines yield high values; defocused or motion-blurred shots yield low values (reference threshold: below 50 considered blurry).
2. **Canny Edge Density:** Percentage of edge pixels, reflecting spine prominence on the rind.

In [ ]:
# 5.3. Measure sharpness (Laplacian variance) and edge density (Canny)
BLUR_THRESHOLD = 50.0

sharpness_records = []
for path in color_sample_paths:
    with Image.open(path) as img:
        img.draft("RGB", (600, 600))
        img = img.convert("RGB")
        img.thumbnail((300, 300))
        gray = cv2.cvtColor(np.asarray(img), cv2.COLOR_RGB2GRAY)
    lap_var = cv2.Laplacian(gray, cv2.CV_64F).var()
    sharpness_records.append({
        "filename": os.path.basename(path),
        "laplacian_variance": round(lap_var, 2),
        "edge_density_pct": round((cv2.Canny(gray, 50, 150) > 0).mean() * 100, 2),
        "is_blurry": lap_var < BLUR_THRESHOLD,
    })

df_sharpness = pd.DataFrame(sharpness_records)
blurry_count = int(df_sharpness["is_blurry"].sum())
print(f"[INFO] Audited {len(df_sharpness)} sample images:")
print(f"  Mean Laplacian variance: {df_sharpness['laplacian_variance'].mean():.2f}")
print(f"  Mean Canny edge density: {df_sharpness['edge_density_pct'].mean():.2f}%")
print(f"  Blurry image count (below {BLUR_THRESHOLD:.0f}): {blurry_count} ({blurry_count / len(df_sharpness):.1%})")
if blurry_count:
    display(df_sharpness[df_sharpness["is_blurry"]].sort_values("laplacian_variance"))

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
sns.histplot(df_sharpness["laplacian_variance"], kde=True, color="indigo", ax=axes[0])
axes[0].axvline(BLUR_THRESHOLD, color="red", linestyle="--", label=f"Blur threshold ({BLUR_THRESHOLD:.0f})")
axes[0].set_title("Laplacian Variance Distribution", fontweight="bold")
axes[0].set_xlabel("Laplacian Variance (higher = sharper)")
axes[0].legend()
sns.histplot(df_sharpness["edge_density_pct"], kde=True, color="forestgreen", ax=axes[1])
axes[1].set_title("Canny Edge Density Distribution", fontweight="bold")
axes[1].set_xlabel("Edge Density (%)")
plt.tight_layout()
plt.show()

## 6. Preprocessing and Data Augmentation Pipeline

Based on the EDA findings in Section 5, the preprocessing pipeline is designed as follows:

| EDA Finding | Pipeline Solution |
| :--- | :--- |
| High-res $6000 \times 4000$ (3:2 aspect ratio) decodes slowly | Read JPEG in draft mode, resize along shortest side to **preserve aspect ratio**, then crop to square |
| Spines and carpel grooves are critical features | Apply CLAHE strictly to L channel (CIE LAB) to enhance local texture without distorting peel hue |
| Significant brightness variations across images | Mild `ColorJitter` (hue delta restricted to $\pm 0.03$ to preserve chromatic ripeness signals) |
| Limited fruit count (132 train fruits), prone to overfitting | Random crop, horizontal flip, rotation; `RandomErasing` forces model to attend to whole fruit surface |
| Raw images are uniformly sharp | `GaussianBlur` applied with low probability to simulate real-world mobile capture degradation |
| Ripening is a continuous physiological spectrum | Mixup/CutMix generate soft targets, directly compatible with soft labels from Teacher distillation |

All components are encapsulated into a **unified `VisionPreprocessor` class**, verified in this notebook, and replicated in `preprocess_vision.py`.

### 6.1. Definition of `VisionPreprocessor` Class

Three augmentation policies are provided for the training set:

| Component | `light` | `medium` (default) | `strong` |
| :--- | :---: | :---: | :---: |
| RandomResizedCrop scale | 0.85 - 1.0 | 0.8 - 1.0 | 0.7 - 1.0 |
| RandomRotation | ±10° | ±15° | ±20° |
| ColorJitter | None | Yes | Yes (stronger) |
| GaussianBlur (prob) | 0 | 0.2 | 0.3 |
| RandomPerspective (prob) | 0 | 0 | 0.3 |
| RandomErasing (prob) | 0 | 0.2 | 0.3 |

Recommendation: Use `light` for smaller architectures or initial baselines, `medium` for general training, and `strong` for data-hungry vision transformers (ViT, Swin) susceptible to overfitting.

In [ ]:
# 6.1. Definition of VisionPreprocessor Class (Replicated in preprocess_vision.py)
class VisionPreprocessor:
    """Durian RGB image preprocessing and data augmentation suite.

    Provides:
    - `get_train_transforms()` / `get_val_transforms()`: torchvision transform pipelines for Dataset.
    - `load_image()`: reads images from file path, numpy array, or PIL Image.
    - `preprocess_image()`: pre-processes a single image into normalized tensor (C, H, W).
    - `apply_clahe()`: isolated CLAHE transform for visualization.
    - `denormalize()`: maps normalized tensor back to uint8 RGB image for rendering.
    - `mix_batch()`: batch-level Mixup / CutMix returning soft targets.
    """

    IMAGENET_MEAN = (0.485, 0.456, 0.406)
    IMAGENET_STD = (0.229, 0.224, 0.225)

    # Configuration for each augmentation policy
    # color_jitter = (brightness, contrast, saturation, hue); None disables jitter
    POLICY_CONFIGS: Dict[str, Dict] = {
        "light": {
            "crop_scale": (0.85, 1.0),
            "rotation": 10,
            "color_jitter": None,
            "blur_prob": 0.0,
            "perspective_prob": 0.0,
            "erasing_prob": 0.0,
        },
        "medium": {
            "crop_scale": (0.8, 1.0),
            "rotation": 15,
            "color_jitter": (0.15, 0.15, 0.15, 0.03),
            "blur_prob": 0.2,
            "perspective_prob": 0.0,
            "erasing_prob": 0.2,
        },
        "strong": {
            "crop_scale": (0.7, 1.0),
            "rotation": 20,
            "color_jitter": (0.2, 0.2, 0.2, 0.04),
            "blur_prob": 0.3,
            "perspective_prob": 0.3,
            "erasing_prob": 0.3,
        },
    }

    class ClaheTransform:
        """Applies CLAHE strictly to the L channel of CIE LAB color space.

        Only modifies lightness L to accentuate spine sutures and pedicel details
        while preserving peel chromaticity (A, B channels remain intact).
        Defined as a class (instead of lambda) to support multiprocessing pickling.
        """

        def __init__(self, clip_limit: float = 2.0, tile_grid_size: Tuple[int, int] = (8, 8)):
            self.clip_limit = clip_limit
            self.tile_grid_size = tuple(tile_grid_size)

        def __call__(self, img: Union[Image.Image, np.ndarray]) -> Image.Image:
            """Receives PIL Image or RGB numpy array, returns illumination-balanced PIL Image."""
            rgb = np.asarray(img.convert("RGB")) if isinstance(img, Image.Image) else img
            l_channel, a_channel, b_channel = cv2.split(cv2.cvtColor(rgb, cv2.COLOR_RGB2LAB))
            clahe = cv2.createCLAHE(clipLimit=self.clip_limit, tileGridSize=self.tile_grid_size)
            lab = cv2.merge((clahe.apply(l_channel), a_channel, b_channel))
            return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

        def __repr__(self) -> str:
            return f"ClaheTransform(clip_limit={self.clip_limit}, tile_grid_size={self.tile_grid_size})"

    def __init__(
        self,
        image_size: Union[int, Tuple[int, int]] = 224,
        use_clahe: bool = True,
        clahe_clip_limit: float = 2.0,
        clahe_tile_grid_size: Tuple[int, int] = (8, 8),
        aug_policy: str = "medium",
        resize_ratio: float = 1.14,
        mean: Optional[Sequence[float]] = None,
        std: Optional[Sequence[float]] = None,
        num_classes: int = 3,
        mixup_alpha: float = 0.8,
        cutmix_alpha: float = 1.0,
        mix_prob: float = 1.0,
    ):
        """Initialize vision preprocessor.

        Args:
            image_size: Output image dimension, int (square) or tuple (H, W).
            use_clahe: Enable/disable CLAHE lightness balancing.
            clahe_clip_limit: Contrast clipping limit for CLAHE.
            clahe_tile_grid_size: Grid partition size for CLAHE.
            aug_policy: Augmentation strength for training: 'light', 'medium', or 'strong'.
            resize_ratio: Shortest-side resize multiplier relative to crop size (default 1.14 ~ 256/224).
            mean: Normalization channel means, defaults to ImageNet.
            std: Normalization channel stds, defaults to ImageNet.
            num_classes: Ripeness class count for soft target generation.
            mixup_alpha: Beta distribution alpha for Mixup (0 to disable).
            cutmix_alpha: Beta distribution alpha for CutMix (0 to disable).
            mix_prob: Probability of applying Mixup/CutMix per batch.
        """
        policy = aug_policy.lower()
        if policy not in self.POLICY_CONFIGS:
            raise ValueError(f"aug_policy must be in {list(self.POLICY_CONFIGS)}, got: '{aug_policy}'")

        self.image_size = (image_size, image_size) if isinstance(image_size, int) else tuple(image_size)
        self.resize_size = int(round(min(self.image_size) * resize_ratio))
        self.use_clahe = use_clahe
        self.aug_policy = policy
        self.mean = tuple(mean) if mean is not None else self.IMAGENET_MEAN
        self.std = tuple(std) if std is not None else self.IMAGENET_STD

        self.num_classes = num_classes
        self.mixup_alpha = mixup_alpha
        self.cutmix_alpha = cutmix_alpha
        self.mix_prob = mix_prob

        self.clahe = self.ClaheTransform(clahe_clip_limit, clahe_tile_grid_size)
        self.train_transforms = self._build_train_transforms()
        self.val_transforms = self._build_val_transforms()

    # ------------------------------------------------------------------
    # Pipeline transforms
    # ------------------------------------------------------------------
    def _build_common_head(self) -> list:
        """Common initial steps: aspect-preserving resize followed by CLAHE."""
        head = [transforms.Resize(self.resize_size, interpolation=transforms.InterpolationMode.BILINEAR)]
        if self.use_clahe:
            head.append(self.clahe)
        return head

    def _build_normalize_tail(self) -> list:
        """Common final steps: ToTensor and normalization with mean/std."""
        return [transforms.ToTensor(), transforms.Normalize(mean=self.mean, std=self.std)]

    def _build_train_transforms(self) -> transforms.Compose:
        """Construct data augmentation pipeline for training set based on `aug_policy`."""
        cfg = self.POLICY_CONFIGS[self.aug_policy]
        steps = self._build_common_head()

        if cfg["perspective_prob"] > 0:
            steps.append(transforms.RandomPerspective(distortion_scale=0.15, p=cfg["perspective_prob"]))

        steps += [
            transforms.RandomResizedCrop(
                self.image_size,
                scale=cfg["crop_scale"],
                ratio=(0.85, 1.15),
                interpolation=transforms.InterpolationMode.BILINEAR,
            ),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomRotation(degrees=cfg["rotation"]),
        ]

        if cfg["color_jitter"] is not None:
            brightness, contrast, saturation, hue = cfg["color_jitter"]
            steps.append(transforms.ColorJitter(brightness, contrast, saturation, hue))

        if cfg["blur_prob"] > 0:
            steps.append(
                transforms.RandomApply([transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 1.2))], p=cfg["blur_prob"])
            )

        steps += self._build_normalize_tail()

        if cfg["erasing_prob"] > 0:
            steps.append(
                transforms.RandomErasing(p=cfg["erasing_prob"], scale=(0.02, 0.2), ratio=(0.3, 3.3), value="random")
            )

        return transforms.Compose(steps)

    def _build_val_transforms(self) -> transforms.Compose:
        """Construct deterministic pipeline for val/test sets: aspect-preserving resize and center crop."""
        steps = self._build_common_head()
        steps.append(transforms.CenterCrop(self.image_size))
        steps += self._build_normalize_tail()
        return transforms.Compose(steps)

    def get_train_transforms(self) -> transforms.Compose:
        """Return training transform pipeline."""
        return self.train_transforms

    def get_val_transforms(self) -> transforms.Compose:
        """Return validation/testing transform pipeline."""
        return self.val_transforms

    # ------------------------------------------------------------------
    # Single-image processing
    # ------------------------------------------------------------------
    def load_image(self, image_input: Union[str, Image.Image, np.ndarray]) -> Image.Image:
        """Load image and return PIL RGB Image."""
        if isinstance(image_input, str):
            if not os.path.exists(image_input):
                raise FileNotFoundError(f"Image file not found: {image_input}")
            img = Image.open(image_input)
            draft_size = self.resize_size * 2
            img.draft("RGB", (draft_size, draft_size))
            return img.convert("RGB")
        if isinstance(image_input, np.ndarray):
            return Image.fromarray(image_input).convert("RGB")
        if isinstance(image_input, Image.Image):
            return image_input.convert("RGB")
        raise TypeError(f"Unsupported image input type: {type(image_input)}")

    def preprocess_image(
        self,
        image_input: Union[str, Image.Image, np.ndarray],
        is_train: bool = False,
    ) -> torch.Tensor:
        """Preprocess a single image, returning normalized tensor (C, H, W)."""
        pipeline = self.train_transforms if is_train else self.val_transforms
        return pipeline(self.load_image(image_input))

    def apply_clahe(self, img: Union[Image.Image, np.ndarray]) -> Image.Image:
        """Apply isolated CLAHE transform, primarily for visualization."""
        return self.clahe(img)

    def denormalize(self, tensor: torch.Tensor) -> np.ndarray:
        """Convert normalized tensor (C, H, W) or (1, C, H, W) back to uint8 RGB numpy array."""
        if tensor.dim() == 4:
            tensor = tensor.squeeze(0)
        mean = torch.tensor(self.mean).view(3, 1, 1)
        std = torch.tensor(self.std).view(3, 1, 1)
        img = (tensor.detach().cpu() * std + mean).clamp(0.0, 1.0)
        return (img.permute(1, 2, 0).numpy() * 255).astype(np.uint8)

    # ------------------------------------------------------------------
    # Batch-level Mixup / CutMix
    # ------------------------------------------------------------------
    def mix_batch(self, images: torch.Tensor, labels: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
        """Apply Mixup or CutMix (random 50/50 selection) to a batch.

        Args:
            images: Image tensor (B, C, H, W) after training transforms.
            labels: Integer class indices (B,) or soft labels (B, num_classes).

        Returns:
            Tuple (augmented images, soft target tensor (B, num_classes)).
        """
        targets = self._to_soft_targets(labels)
        mixup_on, cutmix_on = self.mixup_alpha > 0, self.cutmix_alpha > 0
        if (not mixup_on and not cutmix_on) or torch.rand(1).item() > self.mix_prob:
            return images, targets

        perm = torch.randperm(images.size(0), device=images.device)
        use_cutmix = cutmix_on and (not mixup_on or torch.rand(1).item() < 0.5)

        if use_cutmix:
            lam = float(np.random.beta(self.cutmix_alpha, self.cutmix_alpha))
            y1, x1, y2, x2 = self._random_bbox(images.shape[2], images.shape[3], lam)
            images = images.clone()
            images[:, :, y1:y2, x1:x2] = images[perm, :, y1:y2, x1:x2]
            # Recalculate lambda according to clipped bbox area
            lam = 1.0 - (y2 - y1) * (x2 - x1) / (images.shape[2] * images.shape[3])
        else:
            lam = float(np.random.beta(self.mixup_alpha, self.mixup_alpha))
            images = lam * images + (1.0 - lam) * images[perm]

        return images, lam * targets + (1.0 - lam) * targets[perm]

    def _to_soft_targets(self, labels: torch.Tensor) -> torch.Tensor:
        """Convert class indices to one-hot targets; preserve if already soft targets."""
        if labels.dim() == 2:
            return labels.float()
        return torch.nn.functional.one_hot(labels.long(), self.num_classes).float()

    @staticmethod
    def _random_bbox(height: int, width: int, lam: float) -> Tuple[int, int, int, int]:
        """Generate random cropping bounding box for CutMix, area ~ (1 - lam)."""
        cut_ratio = np.sqrt(1.0 - lam)
        cut_h, cut_w = int(height * cut_ratio), int(width * cut_ratio)
        cy, cx = np.random.randint(height), np.random.randint(width)
        y1, y2 = np.clip([cy - cut_h // 2, cy + cut_h // 2], 0, height)
        x1, x2 = np.clip([cx - cut_w // 2, cx + cut_w // 2], 0, width)
        return int(y1), int(x1), int(y2), int(x2)


preprocessor = VisionPreprocessor(image_size=224, use_clahe=True, aug_policy="medium")
print("[INFO] Train Pipeline:")
print(preprocessor.get_train_transforms())
print("\n[INFO] Val/Test Pipeline:")
print(preprocessor.get_val_transforms())

### 6.2. Validation of CLAHE on L Channel

Per project specifications (Section 1.3.1), CLAHE is employed to accentuate spine textures and the pedicel region:
- Equalizing RGB channels directly alters color ratios and induces peel hue distortion.
- In **CIE LAB** color space, the L channel isolates lightness and surface detail, while A and B represent chrominance. Applying CLAHE strictly to L sharpens spines while keeping peel color signals intact.

The cell below visually and quantitatively evaluates: L channel contrast gain vs. A/B color shift.

In [ ]:
# 6.2. Compare image before and after CLAHE
sample_img_path = df_img["filepath"].iloc[0]
orig_img = preprocessor.load_image(sample_img_path)
orig_img.thumbnail((800, 800))
clahe_img = preprocessor.apply_clahe(orig_img)

np_orig, np_clahe = np.asarray(orig_img), np.asarray(clahe_img)
lab_orig = cv2.cvtColor(np_orig, cv2.COLOR_RGB2LAB).astype(np.float32)
lab_clahe = cv2.cvtColor(np_clahe, cv2.COLOR_RGB2LAB).astype(np.float32)
diff_gray = cv2.cvtColor(cv2.absdiff(np_orig, np_clahe), cv2.COLOR_RGB2GRAY)

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
axes[0].imshow(orig_img)
axes[0].set_title("Original Image", fontweight="bold")
axes[1].imshow(clahe_img)
axes[1].set_title("After CLAHE", fontweight="bold")
heat = axes[2].imshow(diff_gray, cmap="inferno")
axes[2].set_title("Difference Map (Absolute Residuals)", fontweight="bold")
fig.colorbar(heat, ax=axes[2], fraction=0.046, pad=0.04)
for ax in axes[:3]:
    ax.axis("off")
axes[3].hist(lab_orig[:, :, 0].ravel(), bins=50, color="gray", alpha=0.6, label="Original")
axes[3].hist(lab_clahe[:, :, 0].ravel(), bins=50, color="crimson", alpha=0.6, label="CLAHE")
axes[3].set_title("L-Channel Histogram", fontweight="bold")
axes[3].set_xlabel("L value")
axes[3].legend()
plt.tight_layout()
plt.show()

l_std_before, l_std_after = lab_orig[:, :, 0].std(), lab_clahe[:, :, 0].std()
ab_shift = np.abs(lab_clahe[:, :, 1:] - lab_orig[:, :, 1:]).mean()
print(f"[INFO] L-channel standard deviation: {l_std_before:.2f} -> {l_std_after:.2f} ({(l_std_after / l_std_before - 1):+.1%})")
print(f"[INFO] Mean absolute shift across A, B chrominance channels: {ab_shift:.2f} (scale 0-255); peel color is well preserved.")

### 6.3. Validation of Augmentation Policies across Levels

Each row corresponds to an `aug_policy` level. The first column shows the deterministic val/test pipeline output (aspect-preserving resize + center crop); subsequent columns display random stochastic variations from the training pipeline. Visual inspection confirms that fruit morphology, rind coloration, and spine textures remain intact.

In [ ]:
# 6.3. Visualize augmentation across light, medium, and strong policies
NUM_AUG_SAMPLES = 4
policies = list(VisionPreprocessor.POLICY_CONFIGS)

fig, axes = plt.subplots(len(policies), NUM_AUG_SAMPLES + 1, figsize=(4 * (NUM_AUG_SAMPLES + 1), 4 * len(policies)))
for row, policy in enumerate(policies):
    policy_preprocessor = VisionPreprocessor(image_size=224, use_clahe=True, aug_policy=policy)
    axes[row, 0].imshow(policy_preprocessor.denormalize(policy_preprocessor.preprocess_image(orig_img, is_train=False)))
    axes[row, 0].set_title(f"{policy}: val/test", fontweight="bold", color="green")
    for col in range(1, NUM_AUG_SAMPLES + 1):
        aug_tensor = policy_preprocessor.preprocess_image(orig_img, is_train=True)
        axes[row, col].imshow(policy_preprocessor.denormalize(aug_tensor))
        axes[row, col].set_title(f"{policy}: train #{col}")
    for ax in axes[row]:
        ax.axis("off")

plt.suptitle("Augmentation Results by Policy Level", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

### 6.4. Validation of Mixup and CutMix

The ripening progression (`Unripe` → `Ripe` → `Overripe`) represents a continuous physiological spectrum; hard one-hot targets (e.g. `[1, 0, 0]`) can lead to overconfidence near inter-class decision boundaries.

- **Mixup:** Linearly interpolates two images and labels with $\lambda \sim \mathrm{Beta}(\alpha, \alpha)$:
  $$\tilde{x} = \lambda x_i + (1 - \lambda) x_j, \qquad \tilde{y} = \lambda y_i + (1 - \lambda) y_j$$
- **CutMix:** Cuts a patch from image $x_j$ and pastes onto $x_i$; labels are blended proportional to the bounding box area.

Both generate **soft targets**, matching the probability distributions emitted by the Teacher model during knowledge distillation. The `mix_batch()` function accepts both hard class indices and soft target vectors. Mixup/CutMix is active strictly during training iterations.

In [ ]:
# 6.4. Visualize Mixup and CutMix on a sample batch spanning all 3 ripeness classes
demo_rows = (
    df_img[df_img["ripeness"].isin(RIPENESS_ORDER)]
    .groupby("ripeness", group_keys=False)
    .head(1)
    .sort_values("ripeness", key=lambda s: s.map(LABEL_TO_IDX))
)
demo_images = torch.stack([preprocessor.preprocess_image(p, is_train=False) for p in demo_rows["filepath"]])
demo_labels = torch.tensor([LABEL_TO_IDX[r] for r in demo_rows["ripeness"]])

mixers = {
    "Mixup": VisionPreprocessor(image_size=224, mixup_alpha=0.8, cutmix_alpha=0.0),
    "CutMix": VisionPreprocessor(image_size=224, mixup_alpha=0.0, cutmix_alpha=1.0),
}


def format_target(target: torch.Tensor) -> str:
    """Format soft target vector as compact string for plot titles."""
    return "[" + ", ".join(f"{v:.2f}" for v in target.tolist()) + "]"


torch.manual_seed(SEED)
np.random.seed(SEED)
n_cols = len(demo_labels)
fig, axes = plt.subplots(1 + len(mixers), n_cols, figsize=(4.5 * n_cols, 4.5 * (1 + len(mixers))), squeeze=False)

for col in range(n_cols):
    axes[0, col].imshow(preprocessor.denormalize(demo_images[col]))
    axes[0, col].set_title(f"Original: {IDX_TO_LABEL[demo_labels[col].item()]}\n{format_target(preprocessor._to_soft_targets(demo_labels)[col])}")

for row, (name, mixer) in enumerate(mixers.items(), start=1):
    mixed_images, soft_targets = mixer.mix_batch(demo_images, demo_labels)
    for col in range(n_cols):
        axes[row, col].imshow(preprocessor.denormalize(mixed_images[col]))
        axes[row, col].set_title(f"{name}\n{format_target(soft_targets[col])}")

for ax in axes.flatten():
    ax.axis("off")
plt.suptitle("Mixup and CutMix: Blended Images and Soft Targets [Unripe, Ripe, Overripe]", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 7. PyTorch Dataset and DataLoader Encapsulation

The `DurianVisionDataset` class serves subsequent pipeline stages:
- `02_train_unimodal_vision`: Training baseline vision-only models.
- `03_create_final_dataset` and `04_train_teacher_and_student`: Multimodal integration and cross-modal distillation.

The dataset accepts the `df_img` metadata DataFrame constructed in Section 4, filters by partition (`train`, `val`, `test`), skips unlabelled samples, and yields `(image_tensor, label, sample_id, orientation)`. The final cell demonstrates `mix_batch()` invocation within a training step.

In [ ]:
# 7.1. Define DurianVisionDataset and test batch loading
class DurianVisionDataset(Dataset):
    """Durian RGB image Dataset for the vision track.

    Args:
        metadata: DataFrame containing filepath, sample_id, orientation, ripeness (and Split if filtering).
        preprocessor: VisionPreprocessor instance used for image transformation.
        split: Partition name ('train', 'val', 'test'); None to select all.
        is_train: True to apply training augmentation, False for validation/test pipeline.
    """

    def __init__(self, metadata: pd.DataFrame, preprocessor: VisionPreprocessor,
                 split: Optional[str] = None, is_train: bool = False):
        df = metadata if split is None else metadata[metadata["Split"] == split]
        known = df["ripeness"].isin(LABEL_TO_IDX)
        self.num_skipped = int((~known).sum())
        self.samples = [
            (row.filepath, LABEL_TO_IDX[row.ripeness], int(row.sample_id), row.orientation)
            for row in df[known].itertuples()
        ]
        self.preprocessor = preprocessor
        self.is_train = is_train

    def __len__(self) -> int:
        return len(self.samples)

    def __getitem__(self, idx: int):
        path, label, sample_id, orientation = self.samples[idx]
        image = self.preprocessor.preprocess_image(path, is_train=self.is_train)
        return image, torch.tensor(label, dtype=torch.long), sample_id, orientation


has_split = df_img["Split"].notna().any()
for split_name in ["train", "val", "test"]:
    ds = DurianVisionDataset(df_img, preprocessor, split=split_name if has_split else None)
    print(f"[INFO] Partition {split_name:<5}: {len(ds):>5} images (skipped {ds.num_skipped} unlabelled images)")

train_dataset = DurianVisionDataset(df_img, preprocessor, split="train" if has_split else None, is_train=True)
batch_size = min(8, len(train_dataset))
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)

batch_images, batch_labels, batch_ids, batch_oris = next(iter(train_loader))
print(f"\n[INFO] Batch image tensor shape: {tuple(batch_images.shape)} (B, C, H, W)")
print(f"[INFO] Batch label tensor shape: {tuple(batch_labels.shape)}")

# Demonstrate training step: apply Mixup/CutMix on fetched batch
mixed_images, soft_targets = preprocessor.mix_batch(batch_images, batch_labels)
print(f"[INFO] Post-mix_batch shapes: images {tuple(mixed_images.shape)}, soft targets {tuple(soft_targets.shape)}")
print(f"[INFO] Probability sum per sample: {soft_targets.sum(dim=1).tolist()}")

cols = 4
rows = (batch_size + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(16, 4 * rows), squeeze=False)
for i, ax in enumerate(axes.flatten()):
    ax.axis("off")
    if i < batch_size:
        ax.imshow(preprocessor.denormalize(batch_images[i]))
        ax.set_title(f"Fruit #{batch_ids[i].item()} | {batch_oris[i]}\n{IDX_TO_LABEL[batch_labels[i].item()]}")
plt.suptitle(f"Sample Batch from DurianVisionDataset ({batch_size} images)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 8. Conclusions and Deliverables

### EDA Findings Summary
- **Labels and Partitions:** 189 fruits strictly balanced 63/63/63; sample-level split into 132 train, 28 val, 29 test, eliminating inter-split data leakage.
- **Physicochemical Correlation:** Firmness drops sharply and TSS increases markedly as fruits progress from `Unripe` to `Ripe`/`Overripe`, confirming strong physiological grounding for the target labels.
- **Image Quality:** 2,718 images, all $6000 \times 4000$ (3:2 aspect ratio), 0 corrupted files detected; illumination spans a moderate range (109 to 188); high overall sharpness (average Laplacian variance ~274, 0 blurred images).
- **Color Profile:** Dataset RGB channel means are slightly higher than ImageNet (~0.07–0.12 per channel); ImageNet statistics are retained when fine-tuning pretrained architectures.
- **Filename Parsing:** In initial regex passes, 2,162/2,718 images were recognized from standard replicate naming. The remaining 556 images with code IDs (e.g., `A10B8-C2`) are resolved in `01_multimodal_eda_and_data_audit.ipynb` and `02_build_final_dataset.ipynb` via the unified lookup dictionary to guarantee 100% data preservation.

### Validated Preprocessing Pipeline
1. JPEG draft decoding + aspect-preserving shortest-side resize.
2. L-channel CLAHE within CIE LAB color space.
3. Policy-based data augmentation for training (`light` / `medium` / `strong`); deterministic center crop for validation/testing.
4. Channel standardization (ImageNet default).
5. Batch-level Mixup and CutMix via `mix_batch()`, seamlessly accommodating soft teacher supervision.

### Deliverables
The `VisionPreprocessor` class from Section 6.1 is replicated in [preprocess_vision.py](preprocess_vision.py). Subsequent training workflows import it as follows:

```python
from preprocess_vision import VisionPreprocessor

preprocessor = VisionPreprocessor(image_size=224, aug_policy="medium")
train_tf = preprocessor.get_train_transforms()
val_tf = preprocessor.get_val_transforms()

for images, labels in train_loader:
    images, soft_targets = preprocessor.mix_batch(images, labels)
    ...
```